# 15檔精選強勢族群基金策略 (穩定優化版)

## 策略邏輯：
1. **初始持股**：由強勢族群挑選 15 檔起始股票。
2. **維持 15 檔**：優化買入邏輯，確保現金利用率，盡可能維持 15 檔滿倉狀態。
3. **修正說明**：已徹底解決日期對齊與現金預算不足導致持股不滿的 Bug。
4. **績效目標**：透過 15 檔分散投資，追求更穩定的淨值增長與較低的 MDD。

In [ ]:
from finlab import login
import pandas as pd
import numpy as np
from finlab import data
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta
import ast
import json
import warnings
warnings.filterwarnings('ignore')

login('97Y21Yf07Tokqp6rnUxsQKHbc4j+HosTsqE5DNh2oWLA9n+pxaCibJSKUK190ocZ#vip_m')

N_DAYS = 5
MIN_LIQ_PCT = 0.6
TOP_GROUPS = 20
TOP_STOCKS = 50           
PORTFOLIO_SIZE = 15       

STOP_LOSS_PCT = 0.10
FIXED_HOLDING_DAYS = 40
COOLING_OFF_DAYS = 5

# Weights: Removed sync weight
weights = {"ret": 1.5, "turnover": 1.0, "inst": 1.0, "conc": 1.0}

print("正在抓取並對齊資料...")
close = data.get("price:收盤價")
open_ = data.get("price:開盤價")
volume = data.get("price:成交股數")
benchmark = data.get('taiex_total_index:收盤指數')
benchmark = benchmark[~benchmark.index.duplicated(keep='first')]
benchmark_ma200 = benchmark.rolling(200).mean()

# ================= 替換起點 =================

# --- 1. 處理「多重標籤」 (Multi-Label Mapping) ---
print("處理多重題材標籤 (混合動態版：早期回填 + 近期逐日更新)...")
theme_raw = data.get("security_industry_themes").copy()
cat_raw = data.get("security_categories").copy()

# 轉換日期格式
theme_raw['key_date'] = pd.to_datetime(theme_raw['key_date']).dt.normalize()

# 💎 核心魔法：兩段式時空切換 (歷史凍結 -> 現實校正 -> 未來動態)
# 1. 歷史凍結區：擷取 2026-01-01 快照，時空穿越到 2010 年。
# 作用：讓 2012 ~ 2026-04-26 的回測，永遠鎖死在 1/1 的標籤狀態。
snapshot_1_date = pd.to_datetime('2026-01-01')
snapshot_1 = theme_raw[theme_raw['key_date'] <= snapshot_1_date].sort_values('key_date').drop_duplicates('stock_id', keep='last').copy()
snapshot_1['key_date'] = pd.to_datetime('2010-01-01')

# 2. 現實校正點：擷取今天 (2026-04-26) 的「真實最新狀態」，設定在 2026-04-27 觸發。
# 作用：當迴圈跑到 4/27，矩陣會瞬間覆蓋成 Finlab 真正的最新狀態 (把 1/1~4/26 之間所有 Finlab 的隱藏更新一次補齊)。
snapshot_2_date = pd.to_datetime('2026-04-25')
snapshot_2 = theme_raw[theme_raw['key_date'] <= snapshot_2_date].sort_values('key_date').drop_duplicates('stock_id', keep='last').copy()
snapshot_2['key_date'] = pd.to_datetime('2026-04-26')

# 3. 未來動態區：保留今天之後的新更新。
# 作用：4/27 之後的每一天，只要 Finlab 有發佈新標籤，就逐日動態替換。
future_updates = theme_raw[theme_raw['key_date'] > snapshot_2_date].copy()

# 合併三段時空並排序
theme_raw = pd.concat([snapshot_1, snapshot_2, future_updates]).sort_values('key_date')


# 準備名稱與顯示用對照表 (維持最新標籤僅供最終 Dashboard 顯示用)
name_mapper = theme_raw.drop_duplicates("stock_id", keep="last").set_index("stock_id")["name"]
cat_mapper = cat_raw.drop_duplicates("stock_id", keep="last").set_index("stock_id")["category"]

# 提取所有曾經出現過的股票，確立基礎 Universe
theme_all_stocks = theme_raw['stock_id'].unique()

# 解析字串為 List
def parse_themes(x):
    try:
        if isinstance(x, str):
            clean_str = x.replace("[", "").replace("]", "").replace("'", "").replace('"', "")
            return [t.strip() for t in clean_str.split(",") if t.strip() != ""]
        return []
    except:
        return []

theme_raw['theme_list'] = theme_raw['category'].apply(parse_themes)
all_themes = theme_raw['theme_list'].explode().dropna().unique()
all_themes = [t for t in all_themes if t != ""]

# 3. 對齊所有資料的欄位 (寬鬆模式: 以收盤價為主)
print("資料對齊與處理...")
foreign = data.get('institutional_investors_trading_summary:外陸資買賣超股數(不含外資自營商)')
trust = data.get('institutional_investors_trading_summary:投信買賣超股數')
dealer = data.get('institutional_investors_trading_summary:自營商買賣超股數(自行買賣)')
rev_yoy = data.get("monthly_revenue:去年同月增減(%)")

# 定義核心 Universe
common_cols = (close.columns
               .intersection(volume.columns)
               .intersection(theme_all_stocks)
               .intersection(cat_mapper.index)) 

# 加入 adj=True 確保股價有還原
close = data.get("price:收盤價")[common_cols]
open_ = data.get("price:開盤價")[common_cols]
volume = volume[common_cols]
cat_mapper = cat_mapper[common_cols]
name_mapper = name_mapper.reindex(common_cols).fillna("") 

# 寬鬆處理其他數據
trust = trust.reindex(columns=common_cols, fill_value=0)
dealer = dealer.reindex(columns=common_cols, fill_value=0)
foreign = foreign.reindex(columns=common_cols, fill_value=0) 
rev_yoy = rev_yoy.reindex(columns=common_cols)
rev_yoy = rev_yoy.reindex(close.index, method='ffill')

inst_total = trust + dealer
inst_buy_yday = inst_total.shift(2).reindex(close.index)
inst_concentration = (inst_total.shift(2) / volume.replace(0, np.nan)).reindex(close.index)
ma200 = close.rolling(200).mean()

print("計算「動態多重標籤」產業選股訊號 (逐日推進矩陣)...")
ret = close.pct_change(N_DAYS)
turnover = close * volume

selected_stocks_signal = {}
valid_index = close.index.intersection(inst_buy_yday.index)

# 建立動態更新的 Theme Matrix (起初全為 0)
current_theme_matrix = pd.DataFrame(0, index=common_cols, columns=all_themes)
theme_update_idx = 0
theme_raw_records = theme_raw[['key_date', 'stock_id', 'theme_list']].to_dict('records')

for date in valid_index[valid_index >= '2011-12-01']:
    
    # 1. 推進日期：更新當天的標籤
    while theme_update_idx < len(theme_raw_records) and theme_raw_records[theme_update_idx]['key_date'] <= date:
        rec = theme_raw_records[theme_update_idx]
        sid = rec['stock_id']
        if sid in current_theme_matrix.index:
            current_theme_matrix.loc[sid] = 0 # 清除該股票舊標籤
            for t in rec['theme_list']:       # 貼上新標籤
                if t in current_theme_matrix.columns:
                    current_theme_matrix.at[sid, t] = 1
        theme_update_idx += 1

    # 防呆：如果全市場沒標籤就跳過
    if current_theme_matrix.sum().sum() == 0:
        continue
        
    # 2. 計算「當天」的產業指標 (利用矩陣相乘實現極速運算)
    day_ret = ret.loc[date].fillna(0)
    day_turnover = turnover.loc[date].fillna(0)
    day_inst = inst_buy_yday.loc[date].fillna(0)
    day_conc = inst_concentration.loc[date].fillna(0)
    
    # 💎 防護機制：剔除當下還沒有任何股票的「未來幽靈產業」，確保排名分母與歷史完全一致
    active_themes = current_theme_matrix.columns[current_theme_matrix.sum(axis=0) > 0]
    active_matrix = current_theme_matrix[active_themes]
    
    g_sum_ret = day_ret @ active_matrix
    g_sum_turnover = day_turnover @ active_matrix
    g_sum_inst = day_inst @ active_matrix
    g_sum_conc = day_conc @ active_matrix
    
    # 計算有有效資料的股票數量以求平均
    has_data_ret = (~ret.loc[date].isna()).astype(int)
    has_data_conc = (~inst_concentration.loc[date].isna()).astype(int)
    g_count_ret = has_data_ret @ active_matrix
    g_count_conc = has_data_conc @ active_matrix
    
    g_mean_ret = g_sum_ret / g_count_ret.replace(0, np.nan)
    g_mean_conc = g_sum_conc / g_count_conc.replace(0, np.nan)
    
    # 3. 結算當天產業評分
    g_score = (g_mean_ret.rank(pct=True) * weights["ret"] +
               g_sum_turnover.rank(pct=True) * weights["turnover"] +
               g_sum_inst.rank(pct=True) * weights["inst"] +
               g_mean_conc.rank(pct=True) * weights["conc"])
    
    # 找出前 TOP_GROUPS 個強勢產業
    is_top = g_score.rank(ascending=False) <= TOP_GROUPS
    strong_themes = is_top[is_top].index.tolist()

    
    if not strong_themes: continue
    
    # 4. 選出候選股票
    candidates = current_theme_matrix[strong_themes].sum(axis=1)
    stocks_in_groups = candidates[candidates > 0].index.tolist()
    
    try:
        df = pd.DataFrame({
            "ret": ret.loc[date, stocks_in_groups],
            "turnover": turnover.loc[date, stocks_in_groups],
            "inst": inst_buy_yday.loc[date, stocks_in_groups],
            "conc": inst_concentration.loc[date, stocks_in_groups],
            "yoy": rev_yoy.loc[date, stocks_in_groups]
        }).dropna()
    except: continue
    
    if df.empty: continue
    liq_cut = df["turnover"].quantile(1 - MIN_LIQ_PCT)
    df = df[(df["turnover"] >= liq_cut) & (df["inst"] > 0) & (df["conc"] > 0) & (df["yoy"] > 0)]
    
    if df.empty: continue
    df["score"] = (df["ret"].rank(pct=True) * weights["ret"] +
                  df["turnover"].rank(pct=True) * weights["turnover"] +
                  df["inst"].rank(pct=True) * weights["inst"] +
                  df["conc"].rank(pct=True) * weights["conc"])
    
    # MA200 濾網
    current_ma200 = ma200.loc[date]
    current_close = close.loc[date]
    valid_stocks = df.index.intersection(current_ma200.index).intersection(current_close.index)
    df = df.loc[valid_stocks]
    df = df[current_close[df.index] > current_ma200[df.index]]
    
    selected_stocks_signal[date] = df.sort_values("score", ascending=False).head(TOP_STOCKS).index.tolist()

print("訊號計算完成。")

# ================= 替換終點 =================

# --- 終極實戰時序版：收盤判定，次日開盤賣出 ---
print(f"執行回測 (目標持股 {PORTFOLIO_SIZE} 檔，僅限每月 10-15 號進場)...【修正版：次日開盤賣出】")

INITIAL_CAPITAL = 10_000_000
CASH = INITIAL_CAPITAL
PORTFOLIO = []         # 當前持倉
PENDING_EXITS = []     # 標記為「待賣出」的股票列表
TRADE_LOG = []
NAV_HISTORY = []
last_buy_date = {}

backtest_dates = close.index.intersection(valid_index)
backtest_dates = backtest_dates[backtest_dates >= '2012-01-01']

# Fix: Fill NaNs for valuation purposes (to handle suspended stocks in portfolio)
close_ffill = close.ffill()

for i, today in enumerate(backtest_dates):
    yesterday = backtest_dates[i-1] if i > 0 else None
    
    # --- [1] 上午開盤：處理昨晚被標記的「待賣出」股票 ---
    for p in PENDING_EXITS:
        sell_price = open_.at[today, p['stock_id']]
        if pd.isna(sell_price): 
            sell_price = close.at[today, p['stock_id']] # 若無開盤價則以今日收盤代替
            
        revenue = sell_price * p['shares']
        fee = revenue * (0.001425 * 0.1 + 0.003) 
        CASH += (revenue - fee)
        
        TRADE_LOG.append({
            'stock_id': p['stock_id'], 'entry_date': p['entry_date'], 'exit_date': today,
            'entry_price': round(p['entry_price'], 2), 'exit_price': round(sell_price, 2),
            'ret': (revenue - fee - p['cost']) / p['cost'], 'exit_reason': p['reason']
        })
    PENDING_EXITS = [] # 執行完畢，清空待賣清單，此時空位才真正釋放
    
    # --- [2] 上午開盤：進場邏輯 (檢查空位併補貨) ---
    if yesterday is not None:
        market_pass = True
        bm_yesterday = benchmark.at[yesterday, benchmark.columns[0]] if yesterday in benchmark.index else np.nan
        bm_ma_yesterday = benchmark_ma200.at[yesterday, benchmark_ma200.columns[0]] if yesterday in benchmark_ma200.index else np.nan
        if pd.notna(bm_yesterday) and pd.notna(bm_ma_yesterday) and bm_yesterday < bm_ma_yesterday: 
            market_pass = False
        
        is_entry_window = 10 <= today.day <= 15
                
        if market_pass and is_entry_window:
            slots_to_fill = PORTFOLIO_SIZE - len(PORTFOLIO)
            if slots_to_fill > 0:
                signals = selected_stocks_signal.get(yesterday, [])
                for sid in signals:
                    if slots_to_fill <= 0: break
                    if any(p['stock_id'] == sid for p in PORTFOLIO): continue
                    if pd.isna(open_.at[today, sid]): continue
                    
                    curr_ma = ma200.at[yesterday, sid]
                    curr_close = close.at[yesterday, sid]
                    if pd.isna(curr_ma) or curr_close <= curr_ma: 
                        continue
                    
                    lbd = last_buy_date.get(sid)
                    if lbd and (close.index.get_loc(yesterday) - close.index.get_loc(lbd)) <= COOLING_OFF_DAYS: 
                        continue
                    
                    holdings_val = sum(close.at[today, pp['stock_id']] * pp['shares'] if pd.notna(close.at[today, pp['stock_id']]) else pp['entry_price'] * pp['shares'] for pp in PORTFOLIO)
                    target_value = min((CASH + holdings_val) / PORTFOLIO_SIZE, CASH * 0.98)
                    
                    entry_price = open_.at[today, sid]
                    shares = int(target_value / (entry_price * (1 + 0.001425*0.1)))
                    
                    if shares > 0:
                        cost = entry_price * shares * (1 + 0.001425*0.1)
                        CASH -= cost
                        PORTFOLIO.append({
                            'stock_id': sid, 'entry_date': today, 'entry_price': entry_price, 
                            'shares': shares, 'cost': cost, 'entry_idx': i
                        })
                        last_buy_date[sid] = today
                        slots_to_fill -= 1

    # --- [3] 下午收盤：結算淨值與判定明日賣出清單 ---
    # Use close_ffill to handle suspended stocks (avoid NaN in sum)
    current_holdings_value = sum(close_ffill.at[today, p['stock_id']] * p['shares'] for p in PORTFOLIO)
    current_nav = CASH + current_holdings_value
    
    new_active_portfolio = []
    for p in PORTFOLIO:
        curr_price = close.at[today, p['stock_id']]
        exit_reason = None
        if pd.notna(curr_price):
            if curr_price < p['entry_price'] * (1 - STOP_LOSS_PCT): 
                exit_reason = "Stop Loss"
            elif (i - p['entry_idx']) >= FIXED_HOLDING_DAYS: 
                exit_reason = "Time Exit"
        if exit_reason:
            p['reason'] = exit_reason
            PENDING_EXITS.append(p)
        else:
            new_active_portfolio.append(p)
    PORTFOLIO = new_active_portfolio 

    NAV_HISTORY.append({
        'date': today, 'nav': current_nav, 'cash': CASH, 'holdings_count': len(PORTFOLIO)
    })

df_nav = pd.DataFrame(NAV_HISTORY).set_index('date')
print("✅ 實戰模型修正完成：今日收盤觸發，次日開盤賣出。")

# 補回這三行重要的計算，解決 KeyError
df_nav['return'] = df_nav['nav'].pct_change()
df_nav['peak'] = df_nav['nav'].cummax()
df_nav['drawdown'] = (df_nav['nav'] - df_nav['peak']) / df_nav['peak']

# ... (績效指標省略, 直接進 Dashboard 產出) ...
last_data_date = df_nav.index[-1].strftime('%Y-%m-%d')
last_prices = close.iloc[-1]
# 1. 基礎數據
daily_ret = df_nav['nav'].pct_change().dropna()
total_days = (df_nav.index[-1] - df_nav.index[0]).days
if total_days < 1: total_days = 1
# 2. 年化報酬 (CAGR) -> 用於 Dashboard 顯示
# (這是實際口袋裡的錢變多的速度，Dashboard 看這個最準)
ann_ret = (df_nav['nav'].iloc[-1] / df_nav['nav'].iloc[0]) ** (365 / total_days) - 1
# 3. 年化波動率
ann_vol = daily_ret.std() * np.sqrt(252)
# 4. 夏普比率 (Sharpe Ratio) -> 採用業界標準算法
# 分子使用「算術平均 (Arithmetic Mean)」並扣除無風險利率 (假設 2%)
risk_free_rate = 0.02 
ann_arithmetic_mean = daily_ret.mean() * 252
sharpe = (ann_arithmetic_mean - risk_free_rate) / ann_vol if ann_vol != 0 else 0
# 5. 其他指標
ann_downside_deviation = np.sqrt((daily_ret.clip(upper=0)**2).mean()) * np.sqrt(252)
mdd_val = abs(df_nav['drawdown'].min())
calmar = ann_ret / mdd_val if mdd_val != 0 else 0
downside_ret = daily_ret[daily_ret < 0]
ann_downside_vol = downside_ret.std() * np.sqrt(252)
# Sortino 分子通常也建議改用算術平均減無風險，或維持 CAGR 減無風險皆可；這裡維持使用 ann_ret 以保持與 Calmar 的一致性
sortino = (ann_ret - risk_free_rate) / ann_downside_vol if ann_downside_vol != 0 else 0

trade_stats = {"win_rate": 0, "avg_win": 0, "avg_loss": 0, "profit_factor": 0, "total_trades": 0}
if TRADE_LOG:
    rets = [t['ret'] for t in TRADE_LOG]
    wins = [r for r in rets if r > 0]; losses = [r for r in rets if r <= 0]
    trade_stats = {
        "win_rate": round(len(wins)/len(rets)*100, 2), "avg_win": round(np.mean(wins)*100, 2) if wins else 0,
        "avg_loss": round(np.mean(losses)*100, 2) if losses else 0, "profit_factor": round(sum(wins)/abs(sum(losses)), 2) if losses and sum(losses)!=0 else 0,
        "total_trades": len(rets)
    }

curr_holdings_data = []
sector_counter = {}
if PORTFOLIO:
    for p in PORTFOLIO:
        sid = p['stock_id']
        name = name_mapper.get(sid, sid) 
        cat = cat_mapper.get(sid, "其他")
        sector_counter[cat] = sector_counter.get(cat, 0) + 1
        pnl = (last_prices[sid] / p['entry_price'] - 1)
        # 固定名稱顯示：不帶 ID 前綴，因為 UI 有可能自己加
        curr_holdings_data.append({
            "stock_id": sid, "name": f"{name}", "category": cat,
            "entry_date": p['entry_date'].strftime('%Y-%m-%d'),
            "entry_price": round(p['entry_price'], 2), 
            "current_price": round(last_prices[sid], 2),
            "current_date": last_data_date,
            "pnl": round(pnl * 100, 2)
        })
sector_pie = [{"name": k, "value": v} for k, v in sector_counter.items()]

recent_signals_data = []
if 'selected_stocks_signal' in locals() and selected_stocks_signal:
    signal_dates = sorted(selected_stocks_signal.keys())[-5:]
    for d in reversed(signal_dates):
        stocks = selected_stocks_signal[d][:5]
        row_data = {"date": d.strftime("%Y-%m-%d"), "stocks": []}
        for i, sid in enumerate(stocks):
            name = name_mapper.get(sid, "")
            cat = cat_mapper.get(sid, "Unknown")
            row_data["stocks"].append(f"{i+1}. {sid} {name} ({cat})")
        recent_signals_data.append(row_data)

recent_ops = []
seen_buys = set()
if TRADE_LOG:
    for t in TRADE_LOG:
        name = name_mapper.get(t['stock_id'], str(t['stock_id']))
        recent_ops.append({
            "date": t['exit_date'].strftime('%Y-%m-%d'), "action": "賣出",
            "stock_id": t['stock_id'], "name": f"{name}", # 這裡改回只傳 Name
            "price": round(t['exit_price'], 2), "reason": t['exit_reason'],
            "pnl": round(t['ret'] * 100, 2),
            "entry_info": f"({t['entry_date'].strftime('%m/%d')} 以 {round(t['entry_price'], 2)} 買入)"
        })
        key = (t['stock_id'], t['entry_date'])
        if key not in seen_buys:
            recent_ops.append({
                "date": t['entry_date'].strftime('%Y-%m-%d'), "action": "買入",
                "stock_id": t['stock_id'], "name": f"{name}",
                "price": round(t['entry_price'], 2), "reason": "訊號進場",
                "pnl": "-", "entry_info": ""
            })
            seen_buys.add(key)
for p in PORTFOLIO:
    key = (p['stock_id'], p['entry_date'])
    if key not in seen_buys:
        name = name_mapper.get(p['stock_id'], str(p['stock_id']))
        recent_ops.append({
            "date": p['entry_date'].strftime('%Y-%m-%d'), "action": "買入",
            "stock_id": p['stock_id'], "name": f"{name}",
            "price": round(p['entry_price'], 2), "reason": "訊號進場",
            "pnl": "-", "entry_info": ""
        })
        seen_buys.add(key)
recent_ops.sort(key=lambda x: x['date'], reverse=True)

historical_trades = []
if TRADE_LOG:
    for t in sorted(TRADE_LOG, key=lambda x: x['exit_date'], reverse=True)[:50]:
        name = name_mapper.get(t['stock_id'], str(t['stock_id']))
        historical_trades.append({
            "stock_id": t['stock_id'], "name": f"{name}",
            "category": cat_mapper.get(t['stock_id'], "其他"), "entry_date": t['entry_date'].strftime('%Y-%m-%d'),
            "exit_date": t['exit_date'].strftime('%Y-%m-%d'), "entry_price": round(t['entry_price'], 2),
            "exit_price": round(t['exit_price'], 2), "ret": round(t['ret'], 4), "exit_reason": t['exit_reason']
        })

# 8. 補回熱圖分析 (Restored Heatmap)
temp_trades = []
if TRADE_LOG:
    for t in TRADE_LOG:
        temp_trades.append({'sid': t['stock_id'], 'in': pd.to_datetime(t['entry_date']), 'out': pd.to_datetime(t['exit_date'])})
if PORTFOLIO:
    for p in PORTFOLIO:
        temp_trades.append({'sid': p['stock_id'], 'in': pd.to_datetime(p['entry_date']), 'out': pd.to_datetime('2099-12-31')})
df_full_t = pd.DataFrame(temp_trades)

sector_series = pd.Series(cat_mapper)
major_sectors = sector_series.value_counts()[sector_series.value_counts() >= 10].index
monthly_groups = df_nav.resample('M') 
heatmap_data = {}
prev_nav = None
for date, group in monthly_groups:
    if len(group) < 1: continue
    yr, mo = str(date.year), date.month
    start_nav = prev_nav if prev_nav is not None else group['nav'].iloc[0]
    m_ret = (group['nav'].iloc[-1] / start_nav - 1) * 100
    prev_nav = group['nav'].iloc[-1]
    test_day = group.index[-1]
    m_stock_rets = (close.loc[test_day] / close.loc[group.index[0]] - 1)
    val_s = m_stock_rets.index.intersection(cat_mapper.keys())
    m_perf = m_stock_rets[val_s].groupby(cat_mapper).median()
    m_major = m_perf[m_perf.index.isin(major_sectors)]
    market_top = m_major.idxmax() if not m_major.empty else "N/A"
    port_top = "現金"
    if not df_full_t.empty:
        active = df_full_t[(df_full_t['in'] <= test_day) & (df_full_t['out'] >= test_day)]
        if not active.empty:
            port_top = active['sid'].map(cat_mapper).value_counts().idxmax()
    if yr not in heatmap_data: heatmap_data[yr] = {}
    heatmap_data[yr][mo] = {"ret": round(m_ret, 2), "market_top": market_top, "port_top": port_top}

# ...
bm_aligned = benchmark.reindex(df_nav.index).ffill()
bm_col = bm_aligned.columns[0]
avg_drawdown = df_nav['drawdown'].mean()

dashboard_data = {
    "summary": { "last_update": datetime.now().strftime("%Y-%m-%d %H:%M:%S"), "sharpe": round(sharpe, 2), "sortino": round(sortino, 2),"downside_risk": round(ann_downside_deviation * 100, 2), "calmar": round(calmar, 2), "ann_ret": round(ann_ret * 100, 2) },
    "trade_stats": trade_stats,
    "current_holdings": curr_holdings_data,
    "recent_signals": recent_signals_data,
    "trades": historical_trades,
    "operations": recent_ops[:50],
    "sectors": sector_pie,
    "heatmap": heatmap_data, # Restored
    "history": [{ "date": d.strftime("%Y-%m-%d"), "nav": round(v, 2), "benchmark": round(bm_aligned.at[d, bm_col], 2), "mdd": round(m * 100, 2) } for d, v, m in zip(df_nav.index, df_nav['nav'], df_nav['drawdown']) ]
}

js_inner = f"var fundData = {json.dumps(dashboard_data, ensure_ascii=False)};"
template_path = r'c:\Users\teraw_rp58jwl\OneDrive\桌面\量化選股策略\dashboard.html'
with open(template_path, 'r', encoding='utf-8') as f:
    full_html = f.read()
final_html = full_html.replace('<script src="data.js"></script>', f'<script>{js_inner}</script>')
index_path = r'c:\Users\teraw_rp58jwl\OneDrive\桌面\量化選股策略\index.html'
with open(index_path, 'w', encoding='utf-8') as f:
    f.write(final_html)
# GitHub 自動同步
repo_dir = r'c:\Users\teraw_rp58jwl\OneDrive\桌面\量化選股策略'
print("📤 正在同步全功能數據至雲端...")
try:
    os.chdir(repo_dir)
    subprocess.run(["git", "add", "index.html"], check=True, capture_output=True)
    subprocess.run(["git", "commit", "-m", f"Dashboard Benchmark Fix: {datetime.now().strftime('%Y-%m-%d %H:%M')}"], check=True, capture_output=True)
    subprocess.run(["git", "push", "origin", "main"], check=True, capture_output=True)
    print(f"✨ 發布成功！瀏覽網址: https://woody-yiu.github.io/TeraWise-Dashboard/")
except Exception as e:
    print(f"⚠️ 自動發布失敗: {e}")

輸入成功!
正在抓取並對齊資料...


Your version is 1.5.7, please install a newer version.
Use "pip install finlab==1.5.12" to update the latest version.


處理多重題材標籤...
資料對齊與處理 (使用 Reindex 避免掉清單)...
計算「多重標籤」產業選股訊號...
訊號計算完成。
執行回測 (目標持股 15 檔，僅限每月 10-15 號進場)...【修正版：次日開盤賣出】
✅ 實戰模型修正完成：今日收盤觸發，次日開盤賣出。
📤 正在同步全功能數據至雲端...
⚠️ 自動發布失敗: name 'os' is not defined
